[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/40_linear_regression.ipynb)

# 🟡 Medium: Linear Regression

Implement **linear regression** using three different approaches — all in pure PyTorch.

Given data `X` of shape `(N, D)` and targets `y` of shape `(N,)`, find weight `w` of shape `(D,)` and bias `b` (scalar) such that:

$$\hat{y} = Xw + b$$

### Signature
```python
class LinearRegression:
    def closed_form(self, X: Tensor, y: Tensor) -> tuple[Tensor, Tensor]: ...
    def gradient_descent(self, X: Tensor, y: Tensor, lr=0.01, steps=1000) -> tuple[Tensor, Tensor]: ...
    def nn_linear(self, X: Tensor, y: Tensor, lr=0.01, steps=1000) -> tuple[Tensor, Tensor]: ...
```

All methods return `(w, b)` where `w` has shape `(D,)` and `b` has shape `()`.

### Method 1 — Closed-Form (Normal Equation)
Augment X with a ones column, then solve:

$$\theta = (X_{aug}^T X_{aug})^{-1} X_{aug}^T y$$

Or use `torch.linalg.lstsq` / `torch.linalg.solve`.

### Method 2 — Gradient Descent from Scratch
Initialize `w` and `b` to zeros. Repeat for `steps` iterations:
```
pred = X @ w + b
error = pred - y
grad_w = (2/N) * X^T @ error
grad_b = (2/N) * error.sum()
w -= lr * grad_w
b -= lr * grad_b
```

### Method 3 — PyTorch nn.Linear
Create `nn.Linear(D, 1)`, use `nn.MSELoss` and an optimizer (e.g., `torch.optim.SGD`).
After training, extract `w` and `b` from the layer.

### Rules
- All inputs and outputs must be **PyTorch tensors**
- Do **NOT** use numpy or sklearn
- `closed_form` must not use iterative optimization
- `gradient_descent` must manually compute gradients (no `autograd`)
- `nn_linear` should use `torch.nn.Linear` and `loss.backward()`

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 3.1 MB/s eta 0:00:00


In [2]:
import torch
import torch.nn as nn

In [19]:

class LinearRegression:
    def closed_form(self, X: torch.Tensor, y: torch.Tensor):
        """Normal equation: w = (X^T X)^{-1} X^T y"""
        #having y as the target values and error = ||y_pred - y||^2
        #with y_pred = XW + b
        #we want to solve W_best = argmin_w(err)
        #this problem has a closed form solution that is the formula above

        #since linalg.solve(A, b) returns result of Ax = b, solved on x
        #we can use it with A = X^T X and b = X^T y
        #since we also want b inside the weights, we can add a column of ones in X so that y = XW gives x1w1 + x2w2 + ... + b

        X_aug = torch.cat([X, torch.ones(X.shape[0], 1)], dim=1)

        W_full = torch.linalg.solve(X_aug.T @ X_aug, X_aug.T @ y)

        W = W_full[:-1]
        b = W_full[-1]

        return (W, b)

    def gradient_descent(self, X: torch.Tensor, y: torch.Tensor,
                         lr: float = 0.01, steps: int = 1000):
        """Manual gradient descent loop
          X: (N, D)
          W: (D, )
          b: ()
        """
        #Classic GD
        d_in = X.shape[1]
        n = X.shape[0]

        W = torch.normal(0, 1 / (d_in ** 0.5), (d_in,)) #xavier init --> (d_in, )
        b = torch.randn(())

        for _ in range(steps):
          pred = X @ W + b #(N,)
          err = pred - y #(N,)

          loss = torch.mean(torch.square(err)) #MSE

          ##### GRADIENTS
          dW = (2 / n) * X.T @ err #(d_in, ) --> sum over all samples for each feature
          db = torch.sum((2 / n) * err) #() --> sum over all samples (N dim of y) of the err

          W -= lr * dW
          b -= lr * db

        return (W, b)



    def nn_linear(self, X: torch.Tensor, y: torch.Tensor,
                  lr: float = 0.01, steps: int = 1000):
        """Train nn.Linear with autograd"""
        #same as before but done automatically with torch and autograd
        d_in = X.shape[1]
        y = y.unsqueeze(1)
        fcl = nn.Linear(d_in, 1, bias=True)
        loss_fn = nn.MSELoss()
        optimizer = torch.optim.SGD(fcl.parameters(), lr=lr)

        for _ in range(steps):

          optimizer.zero_grad()
          pred = fcl(X)
          loss = loss_fn(pred, y)

          loss.backward()
          optimizer.step()

        return (fcl.weight, fcl.bias)






In [20]:
# 🧪 Debug
torch.manual_seed(42)
X = torch.randn(100, 3)
true_w = torch.tensor([2.0, -1.0, 0.5])
y = X @ true_w + 3.0

model = LinearRegression()

w_cf, b_cf = model.closed_form(X, y)
print(f"Closed-form:  w={w_cf}, b={b_cf.item():.4f}")

w_gd, b_gd = model.gradient_descent(X, y, lr=0.05, steps=2000)
print(f"Grad descent: w={w_gd}, b={b_gd.item():.4f}")

w_nn, b_nn = model.nn_linear(X, y, lr=0.05, steps=2000)
print(f"nn.Linear:    w={w_nn}, b={b_nn.item():.4f}")

print(f"\nTrue:         w={true_w}, b=3.0")

Closed-form:  w=tensor([ 2.0000, -1.0000,  0.5000]), b=3.0000
Grad descent: w=tensor([ 2.0000, -1.0000,  0.5000]), b=3.0000
nn.Linear:    w=Parameter containing:
tensor([[ 2.0000, -1.0000,  0.5000]], requires_grad=True), b=3.0000

True:         w=tensor([ 2.0000, -1.0000,  0.5000]), b=3.0


In [21]:
# ✅ SUBMIT
from torch_judge import check
check("linear_regression")


🧪 Testing: Linear Regression (Medium)
──────────────────────────────────────────────────
  ✅ [1/6] Closed-form returns correct shapes (3.1ms)
  ✅ [2/6] Closed-form finds correct weights (2.5ms)
  ✅ [3/6] Gradient descent converges (158.3ms)
  ✅ [4/6] nn.Linear approach works (571.7ms)
  ✅ [5/6] All three methods agree (1091.6ms)
  ✅ [6/6] Closed-form uses no autograd (0.8ms)
──────────────────────────────────────────────────
  🎉 All 6 tests passed! (1827.9ms total)
  Progress saved. Run status() to see your dashboard.

